In [2]:
import os
import io
import re
import json
import pandas as pd
from pathlib import Path
from datetime import date

# Исследование возможности получения CSV таблиц из фотографий результатов с помощью Mistral OCR API

## Тестовые запуски, первый взгляд на данные

In [ ]:
with open('responses/photo_3433@01-04-2026_11-37-10.json', 'r', encoding='utf8') as jf:
    d = json.loads(jf.read())

In [ ]:
json.loads(d)

In [ ]:
with open('responses/photo_3439@04-04-2026_12-12-28.json', 'r', encoding='utf8') as jf:
    contents = jf.read()
    d = json.loads(contents)

In [ ]:
mdtable = d['pages'][0]['markdown'].replace('|n|', '| \n |')

In [ ]:
with open('test.md', 'w+', encoding='utf8') as f:
    f.write(mdtable)

## Вспомогательные функции для основного пайплайна

In [5]:
def read_json(path):
    with open(path, 'r', encoding='utf8') as jf:
        json_string = json.loads(jf.read())
        if isinstance(json_string, str):
            return json.loads(json_string)
        else:
            return json_string

In [6]:
def set_rounds(df):
    new_columns = ['team', 'total'] + [f'round_{i}' for i in range(1, len(df.columns)-1)]
    return df.set_axis(new_columns, axis=1)

In [7]:
def prep_df(df):
    for i, col in enumerate(df.columns):
        if df[col].astype(str).str.len().mean() > 10:
            return set_rounds(df.iloc[:, i:])
    
    return pd.DataFrame()

In [ ]:
df = pd.read_csv('csvs/photo_3419@27-03-2026_11-44-07.csv')
prep_df(df)

## Первый взгляд на хэдеры

In [ ]:
for p in os.listdir('responses'):
    data = read_json(f'responses/{p}')
    md_content = data['pages'][0]['markdown']
    header = data['pages'][0]['header']
    if 'РАУНД 1' not in md_content:
        continue
    if len(data['pages']) > 1:
        print(p)
    with open(f'raw_mds/{Path(p).stem}.md', 'w+', encoding='utf8') as mdfile:
        mdfile.write(data['pages'][0]['markdown'])
    with open('headers.txt', 'a+', encoding='utf8') as headerfile:
        headerfile.write(f'{p}: {str(header)}')
        headerfile.write('\n===================================\n')

## Экстрация данных из хэдеров

In [4]:
# регулярки для поиска информации
DATE_PATTERN = r'(?:0?[1-9]|[12][0-9]|3[01])\s+(?:ЯНВАРЯ|ФЕВРАЛЯ|МАРТА|АПРЕЛЯ|МАЯ|ИЮНЯ|ИЮЛЯ|АВГУСТА|СЕНТЯБРЯ|ОКТЯБРЯ|НОЯБРЯ|ДЕКАБРЯ)'
MONTHS = {
    'ЯНВАРЯ': 1,
    'ФЕВРАЛЯ': 2,
    'МАРТА': 3,
    'АПРЕЛЯ': 4,
    'МАЯ': 5,
    'ИЮНЯ': 6,
    'ИЮЛЯ': 7,
    'АВГУСТА': 8,
    'СЕНТЯБРЯ': 9,
    'ОКТЯБРЯ': 10,
    'НОЯБРЯ': 11,
    'ДЕКАБРЯ': 12
}

In [ ]:
# Первая версия экстрагирующей функции, только для даты
def get_month_and_day(header):
    if not header:
        return 'EMPTY HEADER'
    
    date = None
    try:
        date = re.findall(DATE_PATTERN, header)[0]
    except IndexError:
        print(header)
        return # TODO: check for this return in function call. If None - write faulty header date not set
    day, month = 0, 0
    for k,v in MONTHS.items():
        if k in date:
            month = v
            day = int(date.replace(k, ''))
            break
    
    return day, month, header.replace(date, '')

In [3]:
# Финальная версия функции
def get_game_info(header):
    info = {} # TODO: turn into dataclass

    if not header:
        return {
            'game_date': '',
            'game_number': '',
            'game_type': '',
            'game_place': '',

        }

    date = None
    new_header = None
    try:
        date = re.findall(DATE_PATTERN, header)[0]
        new_header = header.replace(date, '')
        day, month = 0, 0
        for k,v in MONTHS.items():
            if k in date:
                month = v
                day = int(date.replace(k, ''))
                info['game_date'] = (day, month)
                break
    except IndexError:
        info['game_date'] = 'GAME DATE NOT FOUND'
        new_header = header
    
    try:
        info['game_number'] = re.findall(r'#\s?\d+', new_header)[0]
    except IndexError:
        info['game_number'] = 'GAME NUMBER NOT FOUND'
    
    try:
        info['game_type'] = re.findall(r'\[(\w+(?:\s+\w+)*)\]', new_header)[0]
    except IndexError:
        if len(info['game_number']) > 3:
            info['game_type'] = 'Классика'
        else:
            info['game_type'] = 'GAME TYPE NOT FOUND'


    try:
        info['game_place'] = re.findall(r'\b(?:[A-ZА-ЯЁ]+)\b', new_header)[-1]
    except IndexError:
        info['game_place'] = 'GAME PLACE NOT FOUND'
    
    return info

In [ ]:
# Тестовый запуск
games = []
for p in os.listdir('responses'):
    data = read_json(f'responses/{p}')
    if 'РАУНД 1' not in data['pages'][0]['markdown']:
        continue
    games.append(get_game_info(data['pages'][0]['header']))

with open('games.json', 'w+', encoding='utf8') as jsonfile:
    json.dump(games, jsonfile, indent=4, ensure_ascii=False)
    

In [ ]:
# Основной пайплайн
def pipeline(f):
    data = read_json(f)
    md_content = data['pages'][0]['markdown'].rsplit('|', 1)[0]
    
    # if 'РАУНД 1' not in md_content:
    #     return

    md_content = md_content.rsplit('| --- |', 1)[-1]
    md_content = re.sub(r'\s*\([^)]*\)\s*\n?', '', md_content)
    
    try:
        df = pd.read_table(
            io.StringIO(md_content), 
            sep="|", 
            header=None,
            skipinitialspace=True
        ).dropna(axis=1, how='all').iloc[1:]
    except Exception:
        with open(f'{Path(f).stem}_error.md', 'w+', encoding='utf8') as f:
            f.write(md_content)
    
    if md_content and df.empty:
        md_content = re.sub(r'\bn\b', r'\n', md_content)
        df = pd.read_table(
            io.StringIO(md_content), 
            sep="|", 
            header=None,
            skipinitialspace=True
        ).dropna(axis=1, how='all').iloc[1:]
        if df.empty:
            print(f + ' EMPTY DATAFRAME')
    
    # df = prep_df(df)
    # game_info = get_game_info(data['pages'][0]['header'])
    # for k,v in game_info.items():
    #     if k == 'game_date':
    #         year = int(re.findall('\d{4}', f)[-1])
    #         df[k] = date(year, v[1], v[0]) if isinstance(v, tuple) else v
    #     else:
    #         df[k] = v
    # df.to_csv(f'csvs/{Path(f).stem}.csv', index=False)
    return df

In [15]:
pipeline('responses/photo_18@02-04-2022_12-26-18.json')

,0
1,![img-1.jpeg]Отправляйте друзьям и коллегам
2,![img-2.jpeg]![img-3.jpeg]#КВИЗПЛИЗХАБАРОВСК
3,KEMO
4,KHV.QUIZPLEASE.RU


In [16]:
# Запуск основного пайплайна
for p in os.listdir('responses'):
    d = pipeline(f'responses/{p}')

responses/photo_745@31-08-2023_21-01-38.json EMPTY DATAFRAME
responses/photo_1998@04-12-2024_19-20-24.json EMPTY DATAFRAME
responses/photo_2573@15-07-2025_17-01-37.json EMPTY DATAFRAME
responses/photo_2939@01-11-2025_11-04-20.json EMPTY DATAFRAME
responses/photo_207@05-12-2022_14-01-01.json EMPTY DATAFRAME
responses/photo_1513@03-07-2024_17-00-43.json EMPTY DATAFRAME
responses/photo_905@12-10-2023_19-00-10.json EMPTY DATAFRAME
responses/photo_963@24-10-2023_19-55-21.json EMPTY DATAFRAME
responses/photo_1915@13-11-2024_19-15-20.json EMPTY DATAFRAME
responses/photo_3286@11-02-2026_18-02-56.json EMPTY DATAFRAME
responses/photo_1407@01-06-2024_13-02-46.json EMPTY DATAFRAME


UnboundLocalError: cannot access local variable 'df' where it is not associated with a value